# Policy Influence Map — Geographic Base Layer

Generates a network diagram projected onto a Lambert Conformal map of Europe,
for import into Illustrator.

## Data pipeline

Edge counts are loaded from `../data/influence_edges.json`, which is
auto-exported by `parlawspeechCC_v2.ipynb` whenever the classification pipeline
is run. If that file is absent or empty, the notebook falls back to hardcoded
values verified from the current study outputs.

**To update with a better classifier:** rerun `parlawspeechCC_v2.ipynb`
end-to-end (the export cell writes `data/influence_edges.json` automatically),
then rerun this notebook. The map regenerates — no manual N-value editing needed.

## Fallback values (hardcoded)
- Within-trio N values: exact, from `parlawspeechCC_v2.ipynb` Cell 41 output.
- DE→CH and DE→NL N=3: stated explicitly in the paper text (Cell 42).
- AT outer-ring connections omitted: counts not available (classification CSV empty).

## Outputs
- `output/influence_map_base.svg` — primary Illustrator import (text editable)
- `output/influence_map_base.pdf` — print-ready backup

In [ ]:
%pip install -r product_requirements.txt

In [ ]:
import matplotlib
matplotlib.rcParams['svg.fonttype'] = 'none'  # text stays editable in Illustrator

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json, os, pathlib

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader

os.makedirs('output', exist_ok=True)

In [ ]:
# ── LOAD EDGE DATA ────────────────────────────────────────────────────────────
EDGES_FILE = pathlib.Path('../data/influence_edges.json')

HARDCODED_EDGES = [
    ('DK', 'DE', 20, 'informational'),
    ('DE', 'DK',  3, 'informational'),
    ('DE', 'AT',  2, 'cautionary'),
    ('AT', 'DE',  2, 'informational'),
    ('AT', 'DK',  1, 'informational'),
    ('DE', 'CH',  3, 'informational'),
    ('DE', 'NL',  3, 'cautionary'),
]

if EDGES_FILE.exists() and EDGES_FILE.stat().st_size > 0:
    with open(EDGES_FILE) as f:
        _d = json.load(f)
    EDGES = [
        (e['src'], e['tgt'], int(e['n']), e['mechanism'])
        for e in _d['edges'] if int(e['n']) > 0
    ]
    SOURCE_NOTE = (
        f"Pipeline data · {_d.get('total_analytical_citations','?')} analytical citations · "
        f"{len(EDGES)} edges · rerun parlawspeechCC_v2.ipynb to update"
    )
    print(f"Loaded from pipeline: {EDGES_FILE}")
else:
    EDGES = HARDCODED_EDGES
    SOURCE_NOTE = (
        "Hardcoded fallback — pipeline data not available. "
        "Rerun parlawspeechCC_v2.ipynb with an updated classifier to regenerate."
    )
    print(f"WARNING — using hardcoded values ({EDGES_FILE} not found or empty)")

print(f"\n{len(EDGES)} edges loaded:")
for src, tgt, n, mech in sorted(EDGES, key=lambda e: -e[2]):
    print(f"  {src} → {tgt}:  N = {n}  ({mech})")

# ── GEOGRAPHY ────────────────────────────────────────────────────────────────
LON_LAT = {
    'DE': (10.4, 51.2), 'DK': ( 9.5, 56.3), 'AT': (14.5, 47.5),
    'SE': (15.0, 62.0), 'CH': ( 8.2, 46.8), 'NL': ( 5.3, 52.3),
    'UK': (-1.5, 53.0), 'NO': ( 8.5, 60.5), 'FI': (25.0, 64.0),
    'FR': ( 2.2, 46.2), 'BE': ( 4.5, 50.5), 'PL': (19.0, 52.1),
    'HR': (15.5, 45.1), 'HU': (19.5, 47.2), 'CZ': (15.5, 49.7),
    'ES': (-3.7, 40.4),
}

EU_PARL_LON_LAT = (7.77, 48.60)    # European Parliament plenary chamber, Strasbourg

STUDY_COUNTRIES = {'DE', 'AT', 'DK'}
REFERENCE_NODES = {'SE', 'UK'}      # world leaders with no citation edges — shown for context

used_countries = {c for src, tgt, _, _ in EDGES for c in (src, tgt)} | REFERENCE_NODES
RING = {c: ('inner' if c in STUDY_COUNTRIES else 'outer') for c in used_countries}

_pair_count = {}
for src, tgt, _, _ in EDGES:
    k = tuple(sorted([src, tgt]))
    _pair_count[k] = _pair_count.get(k, 0) + 1
EDGE_RAD = {}
for src, tgt, _, _ in EDGES:
    if _pair_count[tuple(sorted([src, tgt]))] >= 2:
        EDGE_RAD[(src, tgt)] = 0.18

# Informational changed to warm orange — blue was too similar to the PLS country shading
MECH_COLOUR = {
    'informational': '#D47B25',   # warm orange
    'cautionary':    '#D7191C',   # red
    'legitimating':  '#756BB1',   # purple
}

def arrow_lw(n, n_max):
    return 0.5 + 4.0 * (n / n_max) ** 0.5

# ── COUNTRY CATEGORIES ────────────────────────────────────────────────────────
# Trio:         countries directly investigated in this study (DE, AT, DK)
#               — a subset of ParlLawSpeech V2, shown as the deeper blue
# World Leaders: welfare pioneers (AT and DK are also Trio → amber+green hatch)
# PLS:          all ParlLawSpeech V2 countries; trio shown in deeper variant,
#               others (HR, HU, CZ, ES) in the lighter base shade
TRIO_COUNTRIES = {'AT', 'DE', 'DK'}
WORLD_LEADERS  = {'SE', 'DK', 'UK', 'NL', 'CH', 'AT'}
PLS_COUNTRIES  = {'AT', 'DE', 'DK', 'HR', 'HU', 'CZ', 'ES'}

# Natural Earth uses 'GB' for United Kingdom; all other codes match our usage
NE_TO_INTERNAL = {'GB': 'UK'}

TRIO_COLOR    = '#2878BE'   # medium/deep blue  – study trio (ParlLawSpeech V2 subset)
LEADER_COLOR  = '#6DC86D'   # sage green        – world leaders / welfare pioneers
PLS_COLOR     = '#B8D9EF'   # light blue        – other ParlLawSpeech V2 countries

# Flag emoji (Regional Indicator Symbols — preserved in SVG unicode for Illustrator)
FLAGS = {
    'DE': '\U0001F1E9\U0001F1EA',  # 🇩🇪
    'DK': '\U0001F1E9\U0001F1F0',  # 🇩🇰
    'AT': '\U0001F1E6\U0001F1F9',  # 🇦🇹
    'SE': '\U0001F1F8\U0001F1EA',  # 🇸🇪
    'CH': '\U0001F1E8\U0001F1ED',  # 🇨🇭
    'NL': '\U0001F1F3\U0001F1F1',  # 🇳🇱
    'UK': '\U0001F1EC\U0001F1E7',  # 🇬🇧
    'NO': '\U0001F1F3\U0001F1F4',  # 🇳🇴
    'FI': '\U0001F1EB\U0001F1EE',  # 🇫🇮
    'HR': '\U0001F1ED\U0001F1F7',  # 🇭🇷
    'HU': '\U0001F1ED\U0001F1FA',  # 🇭🇺
    'CZ': '\U0001F1E8\U0001F1FF',  # 🇨🇿
    'ES': '\U0001F1EA\U0001F1F8',  # 🇪🇸
}

In [ ]:
from matplotlib.lines import Line2D

# ── PROJECTION ────────────────────────────────────────────────────────────────
PROJ = ccrs.LambertConformal(
    central_longitude=10.0, central_latitude=52.0,
    standard_parallels=(49, 61),
)

fig, ax = plt.subplots(figsize=(14, 13), subplot_kw={'projection': PROJ})
ax.set_extent([-10, 27, 36, 71], crs=ccrs.PlateCarree())
fig.patch.set_facecolor('white')

# ── MAP BASE ──────────────────────────────────────────────────────────────────
ax.add_feature(cfeature.NaturalEarthFeature(
    'physical', 'land', '50m', facecolor='#EFEFEA', edgecolor='none'), zorder=1)
ax.add_feature(cfeature.NaturalEarthFeature(
    'physical', 'ocean', '50m', facecolor='#DDF0F8', edgecolor='none'), zorder=1)

# ── COUNTRY CATEGORY SHADING ──────────────────────────────────────────────────
_shp = shpreader.natural_earth(resolution='50m', category='cultural',
                               name='admin_0_countries')
for _rec in shpreader.Reader(_shp).records():
    _ne_iso = _rec.attributes.get('ISO_A2', '-1')
    if _ne_iso in ('-1', '-99', ''):
        _ne_iso = _rec.attributes.get('ADM0_A3', '??')[:2]
    _iso = NE_TO_INTERNAL.get(_ne_iso, _ne_iso)
    _geom      = _rec.geometry
    _in_trio   = _iso in TRIO_COUNTRIES
    _in_pls    = _iso in PLS_COUNTRIES
    _is_leader = _iso in WORLD_LEADERS

    if not (_in_trio or _in_pls or _is_leader):
        continue

    if _in_trio and _is_leader:
        ax.add_geometries([_geom], ccrs.PlateCarree(),
                          facecolor=TRIO_COLOR, edgecolor='none', zorder=2)
        ax.add_geometries([_geom], ccrs.PlateCarree(),
                          facecolor='none', hatch='////', edgecolor=LEADER_COLOR,
                          linewidth=0.0, zorder=3)
    elif _in_pls and _is_leader:
        ax.add_geometries([_geom], ccrs.PlateCarree(),
                          facecolor=PLS_COLOR, edgecolor='none', zorder=2)
        ax.add_geometries([_geom], ccrs.PlateCarree(),
                          facecolor='none', hatch='////', edgecolor=LEADER_COLOR,
                          linewidth=0.0, zorder=3)
    elif _in_trio:
        ax.add_geometries([_geom], ccrs.PlateCarree(),
                          facecolor=TRIO_COLOR, edgecolor='none', zorder=2)
    elif _is_leader:
        ax.add_geometries([_geom], ccrs.PlateCarree(),
                          facecolor=LEADER_COLOR, edgecolor='none', zorder=2)
    elif _in_pls:
        ax.add_geometries([_geom], ccrs.PlateCarree(),
                          facecolor=PLS_COLOR, edgecolor='none', zorder=2)

# ── BORDERS + COASTLINES ──────────────────────────────────────────────────────
ax.add_feature(cfeature.NaturalEarthFeature(
    'cultural', 'admin_0_countries', '50m',
    facecolor='none', edgecolor='#C8C8C8', linewidth=0.6), zorder=4)
ax.add_feature(cfeature.NaturalEarthFeature(
    'physical', 'coastline', '50m',
    facecolor='none', edgecolor='#999999', linewidth=0.8), zorder=4)

# ── PROJECT POSITIONS ─────────────────────────────────────────────────────────
POS = {}
for country in used_countries:
    if country in LON_LAT:
        xy = PROJ.transform_point(LON_LAT[country][0], LON_LAT[country][1],
                                   ccrs.PlateCarree())
        POS[country] = np.array(xy)

_west_m = PROJ.transform_point(-10, 54, ccrs.PlateCarree())[0]
_east_m = PROJ.transform_point( 27, 54, ccrs.PlateCarree())[0]
NODE_R  = abs(_east_m - _west_m) * 0.031

n_max  = max(e[2] for e in EDGES) if EDGES else 1

CITED_COUNT = {}
for _, tgt, n, _ in EDGES:
    CITED_COUNT[tgt] = CITED_COUNT.get(tgt, 0) + n

# ── LINES ─────────────────────────────────────────────────────────────────────
bidir  = {(src, tgt) for src, tgt, _, _ in EDGES
          if any(e[0] == tgt and e[1] == src for e in EDGES)}
OFFSET = NODE_R * 0.22

for src, tgt, n, mech in EDGES:
    if src not in POS or tgt not in POS:
        continue
    s   = POS[src]
    t   = POS[tgt]
    col = MECH_COLOUR.get(mech, '#777777')
    lw  = arrow_lw(n, n_max)

    direction = t - s
    dist      = np.linalg.norm(direction)
    unit      = direction / dist
    perp      = np.array([-unit[1], unit[0]])

    off   = perp * OFFSET if (src, tgt) in bidir else np.zeros(2)
    start = s + unit * NODE_R + off
    end   = t - unit * NODE_R + off

    ax.plot([start[0], end[0]], [start[1], end[1]],
            color=col, lw=lw, solid_capstyle='round', zorder=5)

    chord_mid = (s + t) / 2
    label_off = perp * max(NODE_R * 0.75, dist * 0.12)
    ax.text(chord_mid[0] + label_off[0], chord_mid[1] + label_off[1],
            f'N = {n}',
            ha='center', va='center',
            fontsize=7.5, color=col, fontweight='bold', zorder=6,
            bbox=dict(boxstyle='round,pad=0.12', facecolor='white',
                      edgecolor='none', alpha=0.82))

# ── NODES ─────────────────────────────────────────────────────────────────────
for country, pos in POS.items():
    ring = RING.get(country, 'outer')
    r    = NODE_R       if ring == 'inner' else NODE_R * 0.78
    fc   = '#FFFFFF'    if ring == 'inner' else '#F5F5F5'
    ec   = '#1A1A1A'    if ring == 'inner' else '#AAAAAA'
    tc   = '#1A1A1A'    if ring == 'inner' else '#888888'
    lw   = 2.5          if ring == 'inner' else 1.2
    fs   = 10           if ring == 'inner' else 8

    theta = np.linspace(0, 2 * np.pi, 60)
    ax.fill(pos[0] + r * np.cos(theta), pos[1] + r * np.sin(theta),
            facecolor=fc, edgecolor=ec, linewidth=lw, zorder=7)

    ax.text(pos[0], pos[1] + r * 0.18, country,
            ha='center', va='center',
            fontsize=fs, fontweight='bold', color=tc, zorder=8)

    flag = FLAGS.get(country, '')
    if flag:
        ax.text(pos[0], pos[1] - r * 0.35, flag,
                ha='center', va='center', fontsize=fs - 1, zorder=8)

    # Total incoming citations — to the right of the circle, black background
    cited_n = CITED_COUNT.get(country, 0)
    ax.text(pos[0] + r + NODE_R * 0.18, pos[1],
            f'cited: {cited_n}',
            ha='left', va='center',
            fontsize=8, color='white', fontweight='bold', zorder=9,
            bbox=dict(boxstyle='round,pad=0.22', facecolor='black',
                      edgecolor='none', alpha=0.88))

# ── FAINT LABELS for ParlLawSpeech countries not yet studied ─────────────────
# These are corpus extension candidates — labelled lightly so they read as context,
# not as active nodes. Their shading already marks them; the italic name reinforces that.
PLS_UNUSED_NAMES = {
    'HR': 'Croatia',
    'HU': 'Hungary',
    'CZ': 'Czech Rep.',
    'ES': 'Spain',
}
for _code, _name in PLS_UNUSED_NAMES.items():
    if _code in LON_LAT:
        _xy = np.array(PROJ.transform_point(
            LON_LAT[_code][0], LON_LAT[_code][1], ccrs.PlateCarree()))
        ax.text(_xy[0], _xy[1], f'{_code}\n{_name}',
                ha='center', va='center',
                fontsize=6.5, color='#5A96BC', alpha=0.80,
                fontweight='normal', style='italic', zorder=6,
                linespacing=1.3)

# ── EU PARLIAMENT MARKER ──────────────────────────────────────────────────────
# Dot coloured same as wider ParlLawSpeech; label to the LEFT (over France).
_eu_xy = np.array(PROJ.transform_point(
    EU_PARL_LON_LAT[0], EU_PARL_LON_LAT[1], ccrs.PlateCarree()))
ax.plot([_eu_xy[0]], [_eu_xy[1]],
        marker='o', markersize=9, color=PLS_COLOR,
        markeredgecolor='#5A8FB5', linestyle='none', zorder=9)
ax.text(_eu_xy[0] - NODE_R * 0.55, _eu_xy[1],
        'EU Parliament',
        ha='right', va='center', fontsize=6.5, color='#2878A0',
        zorder=9,
        bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                  edgecolor='none', alpha=0.70))

# ── LEGEND ────────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=TRIO_COLOR, edgecolor='#AAAAAA', linewidth=0.5,
                   label='Study trio  (DE — ParlLawSpeech V2)'),
    mpatches.Patch(facecolor=TRIO_COLOR, edgecolor=LEADER_COLOR, hatch='////',
                   linewidth=0.5,
                   label='Study trio + World Leader  (AT · DK — ParlLawSpeech V2)'),
    mpatches.Patch(facecolor=LEADER_COLOR, edgecolor='#AAAAAA', linewidth=0.5,
                   label='World Leader / welfare pioneer  (SE · UK · NL · CH)'),
    mpatches.Patch(facecolor=PLS_COLOR, edgecolor='#AAAAAA', linewidth=0.5,
                   label='Other ParlLawSpeech countries  (HR · HU · CZ · ES)'),
]
for mech, col in MECH_COLOUR.items():
    if any(e[3] == mech for e in EDGES):
        label = {
            'informational': 'Informational  (learning / monitoring)',
            'cautionary':    'Cautionary  (warning / laggard-awareness)',
            'legitimating':  'Legitimating  (justification by reference)',
        }.get(mech, mech)
        legend_handles.append(
            mpatches.Patch(facecolor=col, edgecolor='none', label=label))
legend_handles += [
    mpatches.Patch(facecolor='#FFFFFF', edgecolor='#1A1A1A', linewidth=2.0,
                   label='Studied country  (inner ring)'),
    mpatches.Patch(facecolor='#F5F5F5', edgecolor='#AAAAAA', linewidth=1.2,
                   label='Referenced / context country  (outer ring)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=PLS_COLOR,
           markeredgecolor='#5A8FB5', markersize=8,
           label='EU Parliament  (Strasbourg — plenary seat)'),
]

ax.legend(handles=legend_handles, loc='lower left', fontsize=7.0,
          frameon=True, framealpha=0.95, edgecolor='#DDDDDD',
          title='Legend', title_fontsize=8.0, ncol=2)

note_col = '#CC6600' if 'Hardcoded' in SOURCE_NOTE else '#555555'
ax.text(0.02, 0.02, SOURCE_NOTE, transform=ax.transAxes,
        fontsize=5.5, color=note_col, style='italic', va='bottom')
ax.text(0.02, 0.09, 'Arrow width  ∝  √N  ·  "cited: N" = total incoming citations',
        transform=ax.transAxes, fontsize=6.5, color='#999', style='italic')
ax.text(0.02, 0.11, 'Flag emoji: editable in SVG / Illustrator',
        transform=ax.transAxes, fontsize=6, color='#AAAAAA', style='italic')
ax.text(0.02, 0.13,
        'UK: 0 citations in pig-welfare corpus — continental parliament scope',
        transform=ax.transAxes, fontsize=5.5, color='#AAAAAA', style='italic')
ax.text(0.02, 0.15,
        'Italic labels (HR · HU · CZ · ES) = ParlLawSpeech countries not yet studied — extension candidates',
        transform=ax.transAxes, fontsize=5.5, color='#5A96BC', style='italic')

plt.tight_layout()
plt.savefig('output/influence_map_base.svg', format='svg', bbox_inches='tight')
plt.savefig('output/influence_map_base.pdf', format='pdf', bbox_inches='tight')
plt.show()
print('Saved: output/influence_map_base.svg  (primary Illustrator import)')
print('Saved: output/influence_map_base.pdf')